# 09 · Deep Q-Networks
**DEEP RL** · ⏱ 60 min · 🧪 3 exercises · ⭐ 570 XP

*Q-learning with a neural network — manual backprop, experience replay, target networks, Huber loss and Double DQN.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "09-dqn"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

In 2015 a single algorithm learned to play 49 Atari games from raw pixels, many at human
level. Under the hood it was **Q-learning** — the same update you wrote for the cliff — with
a neural network in place of the table, plus two engineering tricks that kept it from blowing
up. This lesson builds that algorithm, **DQN**, from scratch in numpy.

Tile coding needed *you* to design features. A neural network learns its own features: the
hidden layers transform the raw state into a representation where the output layer — a
linear function, just like last lesson — can read off the Q-values.

```text
q̂(s, ·; θ) = W₃ · relu(W₂ · relu(W₁ s + b₁) + b₂) + b₃          one output per action
```

## 01 · A neural network is a function with a gradient

A **multi-layer perceptron** (MLP) alternates linear layers and non-linearities. With a batch
of inputs X (shape B × n_in) one layer computes

```text
H = relu(X W + b)          W: n_in × n_out,   b: n_out
```

Training needs ∂loss/∂(every weight). **Backpropagation** is just the chain rule applied layer
by layer, from the output backwards. If D = ∂L/∂(output of a linear layer) and X was its input:

```text
∂L/∂W = Xᵀ D          ∂L/∂b = Σ_batch D          ∂L/∂X = D Wᵀ
```

and passing back through a ReLU multiplies by its derivative, which is 1 where the input was
positive and 0 elsewhere. That is the whole of it — PyTorch's `loss.backward()` does exactly
this bookkeeping for you, for any graph.

> **💡 TIP** · **Check every gradient numerically**
>
> For any parameter θᵢ, (L(θᵢ + ε) − L(θᵢ − ε)) / 2ε ≈ ∂L/∂θᵢ. If your analytic gradient
> disagrees by more than ~1e-6 (relative) on a small network in float64, the backward pass
> has a bug. Ten minutes of gradient checking saves days of "why doesn't my agent learn?".

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("09-dqn#0", {"q": "A linear layer has input X of shape (32, 4) and weights W of shape (4, 64). Upstream gradient D has shape (32, 64). What is the shape of ∂L/∂W = XᵀD?", "options": ["(32, 64)", "(4, 64)", "(64, 4)", "(32, 4)"], "answer": 1, "explain": "Xᵀ is (4, 32), D is (32, 64), so XᵀD is (4, 64) — the same shape as W, as every gradient must be. The batch dimension is summed out by the matrix product."})

---
### 🧪 Exercise 1 · A neural network from scratch · ⭐ **150 XP**

Deep RL is just RL with a neural network as the function approximator — so let's build one, with **no autograd**.

Implement two methods of `MLP` (Linear → ReLU → … → Linear):

- `forward(x)` — `x` has shape `(batch, n_in)`. For each layer: remember its input in `self._inputs`, compute `h @ W + b`, apply ReLU except on the last layer.
- `backward(dout)` — given ∂loss/∂output, return a list of gradients in the same order as `self.params` (`[W0, b0, W1, b1, …]`). Use **∂L/∂W = inputᵀ·d**, **∂L/∂b = Σ_batch d**, and propagate **d ← (d·Wᵀ) ⊙ [input > 0]**.

The check compares your gradients with **finite differences** — the universal way to debug a backward pass. `Adam` is provided; the demo uses both to fit sin(3x).

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 09-dqn/01-numpy-mlp
import numpy as np
import rlhud


class MLP:
    """A fully connected network: Linear → ReLU → Linear → ReLU → … → Linear.

    net = MLP([4, 64, 64, 2], seed=0)
    y = net.forward(x)             # x: (batch, 4)  ->  y: (batch, 2)
    grads = net.backward(dy)       # dy = ∂loss/∂y, same shape as y
                                   # grads[i] = ∂loss/∂params[i]
    net.params = [W0, b0, W1, b1, W2, b2]   with Wi: (n_in, n_out), bi: (n_out,)
    """

    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            # He initialisation keeps activations at a sensible scale through ReLU layers.
            self.params.append(rng.normal(0.0, np.sqrt(2.0 / n_in), size=(n_in, n_out)))
            self.params.append(np.zeros(n_out))
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        """Compute the output and remember what backward() will need."""
        # TODO
        # h = x; for each layer i: save h (the layer input) in self._inputs,
        # h = h @ W + b, and apply ReLU unless it is the last layer. Return h.
        raise NotImplementedError

    def backward(self, dout):
        """Back-propagate ∂loss/∂output. Returns grads in the same order as self.params."""
        # TODO
        # Walk the layers in reverse with d = ∂loss/∂(layer output):
        #   ∂L/∂W = inputᵀ @ d,   ∂L/∂b = d summed over the batch
        #   then (except for the first layer) d = (d @ Wᵀ) * (input > 0)   ← ReLU derivative
        raise NotImplementedError


class Adam:
    """Adam optimiser. Updates the arrays in `params` in place."""

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            m_hat, v_hat = m / (1 - self.b1 ** self.t), v / (1 - self.b2 ** self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


if __name__ == "__main__":
    # Supervised warm-up: fit y = sin(3x) with mean-squared error. No autograd anywhere —
    # the gradient of the loss w.r.t. the output is (pred − y)·2/N, and backward() does the rest.
    rng = np.random.default_rng(0)
    net = MLP([1, 64, 64, 1], seed=0)
    opt = Adam(net.params, lr=3e-3)
    live = rlhud.chart("MSE loss (log10)", smooth=20, xlabel="step")
    for step in range(2000):
        x = rng.uniform(-2, 2, size=(64, 1))
        y = np.sin(3 * x)
        pred = net.forward(x)
        loss = np.mean((pred - y) ** 2)
        opt.step(net.backward(2 * (pred - y) / len(x)))
        live.add("train", float(np.log10(loss)))
    live.flush()
    xs = np.linspace(-2, 2, 81)[:, None]
    rlhud.plot(np.sin(3 * xs[:, 0]), label="sin(3x)", title="MLP fit")
    rlhud.plot(net.forward(xs)[:, 0], label="MLP", title="MLP fit")
    rlhud.stat("Final MSE", float(loss))

In [ ]:
#@title ✅ **Check** · A neural network from scratch (+150 XP) — tests the code you last ran above
rlcolab.check("09-dqn/01-numpy-mlp")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

forward: `h = x; self._inputs = []; for i in range(self.n_layers): W, b = self.params[2*i], self.params[2*i+1]; self._inputs.append(h); h = h @ W + b` and `np.maximum(h, 0)` if i is not the last layer.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

backward: loop `for i in reversed(range(self.n_layers))`. The gradient for W is `self._inputs[i].T @ d`, for b it is `d.sum(axis=0)`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Before moving to layer i−1: `d = (d @ W.T) * (self._inputs[i] > 0)`. The ReLU that produced layer i's input is exactly where `self._inputs[i] > 0`. Skip this for i = 0.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("09-dqn/01-numpy-mlp")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 09-dqn/01-numpy-mlp
# Reference solution
import numpy as np
import rlhud


class MLP:
    """A fully connected network: Linear → ReLU → Linear → ReLU → … → Linear.

    net = MLP([4, 64, 64, 2], seed=0)
    y = net.forward(x)             # x: (batch, 4)  ->  y: (batch, 2)
    grads = net.backward(dy)       # dy = ∂loss/∂y, same shape as y
                                   # grads[i] = ∂loss/∂params[i]
    net.params = [W0, b0, W1, b1, W2, b2]   with Wi: (n_in, n_out), bi: (n_out,)
    """

    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            # He initialisation keeps activations at a sensible scale through ReLU layers.
            self.params.append(rng.normal(0.0, np.sqrt(2.0 / n_in), size=(n_in, n_out)))
            self.params.append(np.zeros(n_out))
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        """Compute the output and remember what backward() will need."""
        h = np.asarray(x, dtype=float)
        self._inputs = []                    # input to each Linear layer
        for i in range(self.n_layers):
            W, b = self.params[2 * i], self.params[2 * i + 1]
            self._inputs.append(h)
            h = h @ W + b                    # Linear
            if i < self.n_layers - 1:
                h = np.maximum(h, 0.0)       # ReLU on every hidden layer, not on the output
        return h

    def backward(self, dout):
        """Back-propagate ∂loss/∂output. Returns grads in the same order as self.params."""
        grads = [None] * len(self.params)
        d = np.asarray(dout, dtype=float)    # ∂loss/∂(output of layer i)
        for i in reversed(range(self.n_layers)):
            W = self.params[2 * i]
            h_in = self._inputs[i]
            grads[2 * i] = h_in.T @ d        # ∂L/∂W = inputᵀ · upstream
            grads[2 * i + 1] = d.sum(axis=0) # ∂L/∂b = upstream summed over the batch
            if i > 0:
                d = d @ W.T                  # ∂L/∂(layer input)
                d = d * (h_in > 0)           # through the ReLU that produced h_in
        return grads


class Adam:
    """Adam optimiser. Updates the arrays in `params` in place."""

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            m_hat, v_hat = m / (1 - self.b1 ** self.t), v / (1 - self.b2 ** self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


if __name__ == "__main__":
    # Supervised warm-up: fit y = sin(3x) with mean-squared error. No autograd anywhere —
    # the gradient of the loss w.r.t. the output is (pred − y)·2/N, and backward() does the rest.
    rng = np.random.default_rng(0)
    net = MLP([1, 64, 64, 1], seed=0)
    opt = Adam(net.params, lr=3e-3)
    live = rlhud.chart("MSE loss (log10)", smooth=20, xlabel="step")
    for step in range(2000):
        x = rng.uniform(-2, 2, size=(64, 1))
        y = np.sin(3 * x)
        pred = net.forward(x)
        loss = np.mean((pred - y) ** 2)
        opt.step(net.backward(2 * (pred - y) / len(x)))
        live.add("train", float(np.log10(loss)))
    live.flush()
    xs = np.linspace(-2, 2, 81)[:, None]
    rlhud.plot(np.sin(3 * xs[:, 0]), label="sin(3x)", title="MLP fit")
    rlhud.plot(net.forward(xs)[:, 0], label="MLP", title="MLP fit")
    rlhud.stat("Final MSE", float(loss))

## 02 · Why naive deep Q-learning fails

Plug a network into Q-learning and update after every step with the target
r + γ maxₐ′ q̂(s′, a′; θ). It usually diverges or oscillates. Three reasons:

1. **Correlated samples.** Consecutive states are nearly identical. SGD assumes independent
   samples; a stream of similar ones drags the whole network toward the current situation and
   it forgets everything else.
2. **A moving target.** The target uses the same θ we are updating. Each step changes the
   target, which changes the next step — the network chases its own tail.
3. **The deadly triad.** Function approximation + bootstrapping + off-policy learning, all at once.

DQN answers (1) with **experience replay** and (2) with a **target network**.

## 03 · Experience replay

Store every transition (s, a, r, s′, terminated) in a big circular buffer — say the last
50 000 steps — and train on **random minibatches** sampled from it.

- Random sampling breaks temporal correlations: a batch mixes early, late, left, right.
- Each transition is reused many times: much better sample efficiency.
- Q-learning is off-policy, so learning from old ε-greedy data is legitimate.

> **ℹ️ NOTE** · **Store `terminated`, not `done`**
>
> The buffer must remember whether s′ was a *real* terminal state. A CartPole episode cut at
> 500 steps is truncated: the pole is still up and s′ has a future worth bootstrapping from.

### Huber loss

Early on, TD errors can be huge. With squared loss the gradient grows with the error, so one
bad target produces a giant update. The **Huber loss** is quadratic near zero and linear
beyond δ, so its gradient is simply the error **clipped to [−δ, δ]**:

```text
Huber(x) = ½x²  if |x| ≤ δ,     δ(|x| − ½δ)  otherwise          ∂Huber/∂x = clip(x, −δ, δ)
```

---
### 🧪 Exercise 2 · Replay buffer & Huber loss · ⭐ **120 XP**

Two small pieces that make DQN stable.

**`ReplayBuffer`** — a circular buffer backed by pre-allocated numpy arrays.
- `add(s, a, r, s2, terminated)`: write into slot `self.pos`, then advance `pos` modulo `capacity`; `size` grows until it hits `capacity`.
- `sample(batch_size)`: draw indices uniformly from the *filled* part `[0, size)` with `self.rng` and return `(S, A, R, S2, D)`.

**Huber loss** — quadratic for small errors, linear for large ones:
- `huber(x, delta)` = ½x² if |x| ≤ δ, else δ(|x| − ½δ)
- `huber_grad(x, delta)` = its derivative, which is just `clip(x, −δ, δ)`.

The demo shows how replay sampling breaks the correlation between neighbouring states, and plots Huber vs squared loss.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 09-dqn/02-replay-buffer
import numpy as np
import rlhud


class ReplayBuffer:
    """Fixed-size circular buffer of transitions (s, a, r, s', terminated).

    buf = ReplayBuffer(capacity=50_000, obs_dim=4, seed=0)
    buf.add(s, a, r, s2, terminated)
    S, A, R, S2, D = buf.sample(64)     # numpy arrays with a leading batch dimension
    len(buf)                            # number of stored transitions (≤ capacity)

    Storage is pre-allocated numpy arrays — much faster than a Python list of tuples
    once you sample tens of thousands of batches.
    """

    def __init__(self, capacity, obs_dim, seed=0):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.pos = 0          # where the next transition goes
        self.size = 0         # how many slots are filled
        self.rng = np.random.default_rng(seed)

    def add(self, s, a, r, s2, terminated):
        """Store one transition, overwriting the oldest one when full."""
        # TODO: write into slot self.pos, advance self.pos circularly, grow self.size up to capacity
        raise NotImplementedError

    def sample(self, batch_size):
        """Uniformly sample `batch_size` stored transitions (with replacement)."""
        # TODO: random indices in [0, self.size) with self.rng, return the 5 arrays indexed by them
        raise NotImplementedError

    def __len__(self):
        return self.size


def huber(x, delta=1.0):
    """Huber loss, element-wise: ½x² for |x| ≤ δ, δ(|x| − ½δ) beyond."""
    # TODO (np.where is your friend)
    raise NotImplementedError


def huber_grad(x, delta=1.0):
    """d huber / dx, element-wise: x inside [−δ, δ], ±δ outside — i.e. clip(x, −δ, δ)."""
    # TODO
    raise NotImplementedError


if __name__ == "__main__":
    # 1) Why replay? Consecutive CartPole states are almost identical. Compare the correlation
    #    between neighbours in a trajectory with neighbours in a replay batch.
    from rlenvs import CartPole
    env, rng = CartPole(seed=0), np.random.default_rng(0)
    buf = ReplayBuffer(10_000, 4, seed=0)
    s, _ = env.reset()
    traj = []
    for t in range(5000):
        a = int(rng.integers(2))
        s2, r, term, trunc, _ = env.step(a)
        buf.add(s, a, r, s2, term)
        traj.append(s[2])
        s = s2 if not (term or trunc) else env.reset()[0]
    traj = np.array(traj)
    batch_theta = buf.sample(5000)[0][:, 2]
    corr = lambda v: float(np.corrcoef(v[:-1], v[1:])[0, 1])
    rlhud.table([["consecutive steps", f"{corr(traj):.3f}"], ["replay sample", f"{corr(batch_theta):.3f}"]],
                headers=["pole angle θ, neighbour pairs from…", "correlation"], title="Replay breaks correlations")
    rlhud.stat("Transitions stored", len(buf))

    # 2) Huber vs squared loss and their gradients
    xs = np.linspace(-3, 3, 61)
    rlhud.plot(0.5 * xs ** 2, x=xs, label="½x² (MSE)", title="Loss")
    rlhud.plot(huber(xs), x=xs, label="Huber δ=1", title="Loss")
    rlhud.plot(xs, x=xs, label="x (MSE gradient)", title="Gradient")
    rlhud.plot(huber_grad(xs), x=xs, label="Huber gradient", title="Gradient")

In [ ]:
#@title ✅ **Check** · Replay buffer & Huber loss (+120 XP) — tests the code you last ran above
rlcolab.check("09-dqn/02-replay-buffer")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

add: `i = self.pos; self.obs[i] = s; …; self.pos = (self.pos + 1) % self.capacity; self.size = min(self.size + 1, self.capacity)`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

sample: `idx = self.rng.integers(0, self.size, size=batch_size)` and index every array with the same `idx`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

huber: `a = np.abs(x); np.where(a <= delta, 0.5 * x**2, delta * (a - 0.5 * delta))`. huber_grad: `np.clip(x, -delta, delta)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("09-dqn/02-replay-buffer")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 09-dqn/02-replay-buffer
# Reference solution
import numpy as np
import rlhud


class ReplayBuffer:
    """Fixed-size circular buffer of transitions (s, a, r, s', terminated).

    buf = ReplayBuffer(capacity=50_000, obs_dim=4, seed=0)
    buf.add(s, a, r, s2, terminated)
    S, A, R, S2, D = buf.sample(64)     # numpy arrays with a leading batch dimension
    len(buf)                            # number of stored transitions (≤ capacity)

    Storage is pre-allocated numpy arrays — much faster than a Python list of tuples
    once you sample tens of thousands of batches.
    """

    def __init__(self, capacity, obs_dim, seed=0):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.pos = 0          # where the next transition goes
        self.size = 0         # how many slots are filled
        self.rng = np.random.default_rng(seed)

    def add(self, s, a, r, s2, terminated):
        """Store one transition, overwriting the oldest one when full."""
        i = self.pos
        self.obs[i], self.actions[i], self.rewards[i] = s, a, r
        self.next_obs[i], self.terminated[i] = s2, float(terminated)
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        """Uniformly sample `batch_size` stored transitions (with replacement)."""
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (self.obs[idx], self.actions[idx], self.rewards[idx],
                self.next_obs[idx], self.terminated[idx])

    def __len__(self):
        return self.size


def huber(x, delta=1.0):
    """Huber loss, element-wise: ½x² for |x| ≤ δ, δ(|x| − ½δ) beyond."""
    x = np.asarray(x, dtype=float)
    a = np.abs(x)
    return np.where(a <= delta, 0.5 * x * x, delta * (a - 0.5 * delta))


def huber_grad(x, delta=1.0):
    """d huber / dx, element-wise: x inside [−δ, δ], ±δ outside — i.e. clip(x, −δ, δ)."""
    return np.clip(np.asarray(x, dtype=float), -delta, delta)


if __name__ == "__main__":
    # 1) Why replay? Consecutive CartPole states are almost identical. Compare the correlation
    #    between neighbours in a trajectory with neighbours in a replay batch.
    from rlenvs import CartPole
    env, rng = CartPole(seed=0), np.random.default_rng(0)
    buf = ReplayBuffer(10_000, 4, seed=0)
    s, _ = env.reset()
    traj = []
    for t in range(5000):
        a = int(rng.integers(2))
        s2, r, term, trunc, _ = env.step(a)
        buf.add(s, a, r, s2, term)
        traj.append(s[2])
        s = s2 if not (term or trunc) else env.reset()[0]
    traj = np.array(traj)
    batch_theta = buf.sample(5000)[0][:, 2]
    corr = lambda v: float(np.corrcoef(v[:-1], v[1:])[0, 1])
    rlhud.table([["consecutive steps", f"{corr(traj):.3f}"], ["replay sample", f"{corr(batch_theta):.3f}"]],
                headers=["pole angle θ, neighbour pairs from…", "correlation"], title="Replay breaks correlations")
    rlhud.stat("Transitions stored", len(buf))

    # 2) Huber vs squared loss and their gradients
    xs = np.linspace(-3, 3, 61)
    rlhud.plot(0.5 * xs ** 2, x=xs, label="½x² (MSE)", title="Loss")
    rlhud.plot(huber(xs), x=xs, label="Huber δ=1", title="Loss")
    rlhud.plot(xs, x=xs, label="x (MSE gradient)", title="Gradient")
    rlhud.plot(huber_grad(xs), x=xs, label="Huber gradient", title="Gradient")

## 04 · Target networks

Keep a second copy of the network, θ⁻, frozen most of the time. Targets use it:

```text
yᵢ = rᵢ + γ (1 − doneᵢ) · maxₐ′ q̂(s′ᵢ, a′; θ⁻)
```

Every C steps (e.g. 500), copy θ⁻ ← θ. Between syncs the regression problem is stationary: the
network fits fixed labels, like supervised learning. A common alternative is a "soft" update
θ⁻ ← τθ + (1 − τ)θ⁻ every step with τ ≈ 0.005.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("09-dqn#1", {"q": "What problem does the target network mainly solve?", "options": ["Correlation between consecutive samples", "Targets that move every time the network is updated", "Overestimation of Q-values", "Lack of exploration"], "answer": 1, "explain": "Replay fixes correlation; the frozen target network keeps the regression labels still for C steps. (Overestimation is Double DQN's job.)"})

## 05 · The full algorithm

```python
for t in range(total_steps):
    eps = epsilon_by_step(t)                        # 1.0 → 0.05 over the first 10k steps
    a = random action if rng.random() < eps else argmax q(s; θ)
    s2, r, terminated, truncated, _ = env.step(a)
    buffer.add(s, a, r, s2, terminated)
    if t >= learning_starts:
        S, A, R, S2, D = buffer.sample(64)
        y = R + γ (1 − D) max q(S2; θ⁻)             # targets from the frozen network
        loss = mean Huber(q(S; θ)[A] − y)           # only the taken actions get gradient
        θ ← Adam step on ∇loss
    if t % C == 0: θ⁻ ← θ
```

The **ε schedule** matters more than it looks: the agent starts knowing nothing, so it should
explore almost uniformly, then lean on its estimates as they improve. A linear decay from 1.0 to
0.05 over the first ~20% of training is a robust default. Try it below — at ε = 0.05 how much
probability does a non-greedy action still get?

In [ ]:
#@title 🎮 **Interactive** · ε-greedy distribution
rlcolab.widget("epsilon", values=[1.2, 0.4], epsilon=0.05)

## 06 · Double DQN

The max in the target is **biased upward**: if Q-estimates are noisy, maxₐ picks whichever
action's noise happened to be positive. Over many bootstraps the overestimation compounds.

**Double DQN** separates *choosing* the next action from *evaluating* it:

```text
y = r + γ (1 − done) · q̂(s′, argmaxₐ′ q̂(s′, a′; θ), θ⁻)
```

The online network picks a′; the target network scores it. Their errors are partly
independent, so the bias shrinks. It is a one-line change and almost always helps.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("09-dqn#2", {"q": "Q_online(s′) = [3, 1], Q_target(s′) = [2, 5], r = 1, γ = 0.5, not terminal. What is the Double DQN target?", "options": ["3.5", "2.0", "2.5", "1.5"], "answer": 1, "explain": "The online net picks a′ = 0 (value 3). The target net evaluates action 0: 2. y = 1 + 0.5·2 = 2.0. Plain DQN would use max Q_target = 5 → 3.5."})

---
### 🧪 Exercise 3 · DQN on CartPole · ⭐ **300 XP**

Put it all together: a Deep Q-Network in pure numpy.

The network, Adam, replay buffer and the training loop are provided (read the loop — it is the whole algorithm on one screen). You implement the four RL pieces it calls:

1. `epsilon_by_step(t, start, end, decay_steps)` — linear decay from `start` to `end`, flat afterwards.
2. `select_action(q_net, obs, epsilon, rng)` — ε-greedy on `q_net.forward(obs[None, :])[0]`.
3. `td_targets(rewards, terminated, q_next_target, gamma, q_next_online=None)` — **y = r + γ(1 − done) max Q_target(s′, ·)**; with `q_next_online` given, use **Double DQN**: the online net chooses a′, the target net scores it.
4. `dqn_loss_and_grad(q_pred, actions, targets)` — mean **Huber** loss on Q(s, a_taken) − y and its gradient w.r.t. `q_pred` (zero for untaken actions).

The check unit-tests each piece, then trains for 70 000 steps (≈ 30 s here) and requires a greedy mean return ≥ 150 (max is 500). Run the demo to watch the learning curve — expect it to be bumpy.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 09-dqn/03-dqn-cartpole
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: network, optimiser, replay
class MLP:
    """Linear → ReLU → … → Linear, with manual backprop (exercise 1)."""

    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.params += [rng.normal(0.0, np.sqrt(2.0 / n_in), size=(n_in, n_out)), np.zeros(n_out)]
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        h, self._inputs = np.asarray(x, dtype=float), []
        for i in range(self.n_layers):
            self._inputs.append(h)
            h = h @ self.params[2 * i] + self.params[2 * i + 1]
            if i < self.n_layers - 1:
                h = np.maximum(h, 0.0)
        return h

    def backward(self, dout):
        """Gradients for the inputs of the MOST RECENT forward() call."""
        grads, d = [None] * len(self.params), np.asarray(dout, dtype=float)
        for i in reversed(range(self.n_layers)):
            grads[2 * i], grads[2 * i + 1] = self._inputs[i].T @ d, d.sum(axis=0)
            if i > 0:
                d = (d @ self.params[2 * i].T) * (self._inputs[i] > 0)
        return grads

    def copy_from(self, other):
        """Overwrite these weights with a copy of `other`'s (used for the target network)."""
        for p, q in zip(self.params, other.params):
            p[...] = q


class Adam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)


class ReplayBuffer:
    """Circular buffer (exercise 2)."""

    def __init__(self, capacity, obs_dim, seed=0):
        self.capacity, self.pos, self.size = capacity, 0, 0
        self.obs = np.zeros((capacity, obs_dim)); self.next_obs = np.zeros((capacity, obs_dim))
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity); self.terminated = np.zeros(capacity)
        self.rng = np.random.default_rng(seed)

    def add(self, s, a, r, s2, terminated):
        i = self.pos
        self.obs[i], self.actions[i], self.rewards[i], self.next_obs[i], self.terminated[i] = s, a, r, s2, terminated
        self.pos, self.size = (i + 1) % self.capacity, min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return self.obs[idx], self.actions[idx], self.rewards[idx], self.next_obs[idx], self.terminated[idx]

    def __len__(self):
        return self.size


# ---------------------------------------------------------------- your code: the DQN pieces
def epsilon_by_step(t, start=1.0, end=0.05, decay_steps=10_000):
    """Linear ε schedule: `start` at t = 0, `end` from t = decay_steps on."""
    # TODO
    raise NotImplementedError


def select_action(q_net, obs, epsilon, rng):
    """ε-greedy on the network's Q-values for a single observation."""
    # TODO: with probability epsilon a random action (rng.random(), rng.integers(n_actions)),
    #       otherwise argmax of q_net.forward(obs[None, :])[0]. n_actions = q_net.params[-1].shape[0]
    raise NotImplementedError


def td_targets(rewards, terminated, q_next_target, gamma, q_next_online=None):
    """Bootstrapped targets y for a batch.

    rewards, terminated : (B,)
    q_next_target       : (B, n_actions) — target network's Q(s′, ·)
    q_next_online       : (B, n_actions) or None — online network's Q(s′, ·)

    DQN:         y = r + γ (1 − done) · max_a Q_target(s′, a)
    Double DQN:  y = r + γ (1 − done) · Q_target(s′, argmax_a Q_online(s′, a))   (when q_next_online is given)
    """
    # TODO: pick next_v per row (max for DQN; target's value at the online argmax for Double DQN),
    #       then y = r + γ (1 − terminated) next_v
    raise NotImplementedError


def dqn_loss_and_grad(q_pred, actions, targets, delta=1.0):
    """Mean Huber loss between Q(s, a) for the TAKEN actions and the targets.

    q_pred  : (B, n_actions) online network output for the batch states
    actions : (B,) ints;  targets : (B,)
    returns (loss, dq) where dq = ∂loss/∂q_pred has q_pred's shape and is zero
    for every action that was not taken.
    """
    # TODO
    # 1) err = Q(s, a_taken) − target        (use q_pred[np.arange(B), actions])
    # 2) loss = mean Huber(err)
    # 3) dq = zeros like q_pred; at the taken actions put clip(err, −δ, δ) / B
    raise NotImplementedError


# ---------------------------------------------------------------- provided: training loop
def evaluate(q_net, episodes=10, seed=1000):
    """Mean return of the GREEDY policy (no exploration) over fixed start states."""
    env, total = CartPole(), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            obs, r, term, trunc, _ = env.step(int(np.argmax(q_net.forward(obs[None, :])[0])))
            total += r
            done = term or trunc
    return total / episodes


def train(total_steps=70_000, seed=0, gamma=0.99, lr=1e-3, batch_size=64, buffer_size=50_000,
          learning_starts=1_000, target_sync=500, eps_decay=10_000, hidden=64, double=True,
          eval_every=2_000, log=None):
    """Train DQN on CartPole. Returns (best_q_net, episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    q = MLP([4, hidden, hidden, 2], seed=seed)
    q_target = MLP([4, hidden, hidden, 2], seed=seed + 1)
    q_target.copy_from(q)
    opt = Adam(q.params, lr=lr)
    buf = ReplayBuffer(buffer_size, 4, seed=seed)
    best, best_score = MLP([4, hidden, hidden, 2]), -1.0

    obs, _ = env.reset(seed=seed)
    ep_return, returns = 0.0, []
    for t in range(total_steps):
        # 1) act ε-greedily and store the transition
        eps = epsilon_by_step(t, decay_steps=eps_decay)
        a = select_action(q, obs, eps, rng)
        obs2, r, terminated, truncated, _ = env.step(a)
        buf.add(obs, a, r, obs2, terminated)      # store `terminated`, NOT truncated
        obs, ep_return = obs2, ep_return + r
        if terminated or truncated:
            returns.append(ep_return)
            if log:
                log(len(returns), ep_return, eps)
            obs, _ = env.reset()
            ep_return = 0.0

        # 2) learn from a random minibatch of past experience
        if t >= learning_starts:
            S, A, R, S2, D = buf.sample(batch_size)
            q_next_t = q_target.forward(S2)
            q_next_o = q.forward(S2) if double else None
            y = td_targets(R, D, q_next_t, gamma, q_next_o)
            q_pred = q.forward(S)                 # forward(S) LAST, so backward() uses S
            loss, dq = dqn_loss_and_grad(q_pred, A, y)
            opt.step(q.backward(dq))

        # 3) every so often, copy the online weights into the frozen target network
        if t % target_sync == 0:
            q_target.copy_from(q)

        # 4) deep RL is noisy: keep the best greedy policy we have seen ("checkpointing")
        if t >= learning_starts and t % eval_every == 0 and returns and np.mean(returns[-5:]) > 100:
            score = evaluate(q, 5, seed=10_000 + t)
            if score >= best_score:
                best_score = score
                best.copy_from(q)
    if best_score < 0 or evaluate(q, 5, seed=99_999) > best_score:
        best.copy_from(q)
    return best, returns


if __name__ == "__main__":
    live = rlhud.chart("DQN on CartPole — episode return", smooth=20, xlabel="episode")
    eps_chart = rlhud.chart("ε (exploration)", xlabel="episode")

    def log(ep, G, eps):
        live.add("return", G)
        eps_chart.add("ε", eps)

    q, returns = train(total_steps=70_000, seed=0, log=log)
    live.flush(); eps_chart.flush()
    score = evaluate(q, 10)
    rlhud.stat("Greedy evaluation (10 episodes)", score, good=score >= 150)
    rlhud.stat("Episodes trained", len(returns))
    # What did it learn? Q(s, ·) as the pole angle sweeps from left to right (cart centred, still).
    thetas = np.linspace(-0.2, 0.2, 41)
    Q = q.forward(np.array([[0.0, 0.0, th, 0.0] for th in thetas]))
    rlhud.plot(Q[:, 0], x=thetas, label="Q(push left)", title="Q-values vs pole angle θ", xlabel="θ (rad)")
    rlhud.plot(Q[:, 1], x=thetas, label="Q(push right)", title="Q-values vs pole angle θ", xlabel="θ (rad)")

In [ ]:
#@title ✅ **Check** · DQN on CartPole (+300 XP) — tests the code you last ran above
rlcolab.check("09-dqn/03-dqn-cartpole")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

epsilon_by_step: `frac = min(t / decay_steps, 1.0); return start + frac * (end - start)`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

td_targets: plain DQN `next_v = q_next_target.max(axis=1)`; Double DQN `a = q_next_online.argmax(axis=1); next_v = q_next_target[np.arange(len(a)), a]`. Then `rewards + gamma * (1 - terminated) * next_v`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

dqn_loss_and_grad: `rows = np.arange(B); err = q_pred[rows, actions] - targets`; loss is the mean of the Huber of err; `dq = np.zeros_like(q_pred); dq[rows, actions] = np.clip(err, -1, 1) / B`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("09-dqn/03-dqn-cartpole")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 09-dqn/03-dqn-cartpole
# Reference solution
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: network, optimiser, replay
class MLP:
    """Linear → ReLU → … → Linear, with manual backprop (exercise 1)."""

    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.params += [rng.normal(0.0, np.sqrt(2.0 / n_in), size=(n_in, n_out)), np.zeros(n_out)]
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        h, self._inputs = np.asarray(x, dtype=float), []
        for i in range(self.n_layers):
            self._inputs.append(h)
            h = h @ self.params[2 * i] + self.params[2 * i + 1]
            if i < self.n_layers - 1:
                h = np.maximum(h, 0.0)
        return h

    def backward(self, dout):
        """Gradients for the inputs of the MOST RECENT forward() call."""
        grads, d = [None] * len(self.params), np.asarray(dout, dtype=float)
        for i in reversed(range(self.n_layers)):
            grads[2 * i], grads[2 * i + 1] = self._inputs[i].T @ d, d.sum(axis=0)
            if i > 0:
                d = (d @ self.params[2 * i].T) * (self._inputs[i] > 0)
        return grads

    def copy_from(self, other):
        """Overwrite these weights with a copy of `other`'s (used for the target network)."""
        for p, q in zip(self.params, other.params):
            p[...] = q


class Adam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)


class ReplayBuffer:
    """Circular buffer (exercise 2)."""

    def __init__(self, capacity, obs_dim, seed=0):
        self.capacity, self.pos, self.size = capacity, 0, 0
        self.obs = np.zeros((capacity, obs_dim)); self.next_obs = np.zeros((capacity, obs_dim))
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity); self.terminated = np.zeros(capacity)
        self.rng = np.random.default_rng(seed)

    def add(self, s, a, r, s2, terminated):
        i = self.pos
        self.obs[i], self.actions[i], self.rewards[i], self.next_obs[i], self.terminated[i] = s, a, r, s2, terminated
        self.pos, self.size = (i + 1) % self.capacity, min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return self.obs[idx], self.actions[idx], self.rewards[idx], self.next_obs[idx], self.terminated[idx]

    def __len__(self):
        return self.size


# ---------------------------------------------------------------- your code: the DQN pieces
def epsilon_by_step(t, start=1.0, end=0.05, decay_steps=10_000):
    """Linear ε schedule: `start` at t = 0, `end` from t = decay_steps on."""
    frac = min(t / decay_steps, 1.0)
    return start + frac * (end - start)


def select_action(q_net, obs, epsilon, rng):
    """ε-greedy on the network's Q-values for a single observation."""
    if rng.random() < epsilon:
        return int(rng.integers(q_net.params[-1].shape[0]))
    return int(np.argmax(q_net.forward(obs[None, :])[0]))


def td_targets(rewards, terminated, q_next_target, gamma, q_next_online=None):
    """Bootstrapped targets y for a batch.

    rewards, terminated : (B,)
    q_next_target       : (B, n_actions) — target network's Q(s′, ·)
    q_next_online       : (B, n_actions) or None — online network's Q(s′, ·)

    DQN:         y = r + γ (1 − done) · max_a Q_target(s′, a)
    Double DQN:  y = r + γ (1 − done) · Q_target(s′, argmax_a Q_online(s′, a))   (when q_next_online is given)
    """
    if q_next_online is None:
        next_v = q_next_target.max(axis=1)
    else:
        best = q_next_online.argmax(axis=1)
        next_v = q_next_target[np.arange(len(best)), best]
    return rewards + gamma * (1.0 - terminated) * next_v


def dqn_loss_and_grad(q_pred, actions, targets, delta=1.0):
    """Mean Huber loss between Q(s, a) for the TAKEN actions and the targets.

    q_pred  : (B, n_actions) online network output for the batch states
    actions : (B,) ints;  targets : (B,)
    returns (loss, dq) where dq = ∂loss/∂q_pred has q_pred's shape and is zero
    for every action that was not taken.
    """
    B = len(actions)
    rows = np.arange(B)
    err = q_pred[rows, actions] - targets
    a = np.abs(err)
    loss = float(np.mean(np.where(a <= delta, 0.5 * err ** 2, delta * (a - 0.5 * delta))))
    dq = np.zeros_like(q_pred)
    dq[rows, actions] = np.clip(err, -delta, delta) / B
    return loss, dq


# ---------------------------------------------------------------- provided: training loop
def evaluate(q_net, episodes=10, seed=1000):
    """Mean return of the GREEDY policy (no exploration) over fixed start states."""
    env, total = CartPole(), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            obs, r, term, trunc, _ = env.step(int(np.argmax(q_net.forward(obs[None, :])[0])))
            total += r
            done = term or trunc
    return total / episodes


def train(total_steps=70_000, seed=0, gamma=0.99, lr=1e-3, batch_size=64, buffer_size=50_000,
          learning_starts=1_000, target_sync=500, eps_decay=10_000, hidden=64, double=True,
          eval_every=2_000, log=None):
    """Train DQN on CartPole. Returns (best_q_net, episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    q = MLP([4, hidden, hidden, 2], seed=seed)
    q_target = MLP([4, hidden, hidden, 2], seed=seed + 1)
    q_target.copy_from(q)
    opt = Adam(q.params, lr=lr)
    buf = ReplayBuffer(buffer_size, 4, seed=seed)
    best, best_score = MLP([4, hidden, hidden, 2]), -1.0

    obs, _ = env.reset(seed=seed)
    ep_return, returns = 0.0, []
    for t in range(total_steps):
        # 1) act ε-greedily and store the transition
        eps = epsilon_by_step(t, decay_steps=eps_decay)
        a = select_action(q, obs, eps, rng)
        obs2, r, terminated, truncated, _ = env.step(a)
        buf.add(obs, a, r, obs2, terminated)      # store `terminated`, NOT truncated
        obs, ep_return = obs2, ep_return + r
        if terminated or truncated:
            returns.append(ep_return)
            if log:
                log(len(returns), ep_return, eps)
            obs, _ = env.reset()
            ep_return = 0.0

        # 2) learn from a random minibatch of past experience
        if t >= learning_starts:
            S, A, R, S2, D = buf.sample(batch_size)
            q_next_t = q_target.forward(S2)
            q_next_o = q.forward(S2) if double else None
            y = td_targets(R, D, q_next_t, gamma, q_next_o)
            q_pred = q.forward(S)                 # forward(S) LAST, so backward() uses S
            loss, dq = dqn_loss_and_grad(q_pred, A, y)
            opt.step(q.backward(dq))

        # 3) every so often, copy the online weights into the frozen target network
        if t % target_sync == 0:
            q_target.copy_from(q)

        # 4) deep RL is noisy: keep the best greedy policy we have seen ("checkpointing")
        if t >= learning_starts and t % eval_every == 0 and returns and np.mean(returns[-5:]) > 100:
            score = evaluate(q, 5, seed=10_000 + t)
            if score >= best_score:
                best_score = score
                best.copy_from(q)
    if best_score < 0 or evaluate(q, 5, seed=99_999) > best_score:
        best.copy_from(q)
    return best, returns


if __name__ == "__main__":
    live = rlhud.chart("DQN on CartPole — episode return", smooth=20, xlabel="episode")
    eps_chart = rlhud.chart("ε (exploration)", xlabel="episode")

    def log(ep, G, eps):
        live.add("return", G)
        eps_chart.add("ε", eps)

    q, returns = train(total_steps=70_000, seed=0, log=log)
    live.flush(); eps_chart.flush()
    score = evaluate(q, 10)
    rlhud.stat("Greedy evaluation (10 episodes)", score, good=score >= 150)
    rlhud.stat("Episodes trained", len(returns))
    # What did it learn? Q(s, ·) as the pole angle sweeps from left to right (cart centred, still).
    thetas = np.linspace(-0.2, 0.2, 41)
    Q = q.forward(np.array([[0.0, 0.0, th, 0.0] for th in thetas]))
    rlhud.plot(Q[:, 0], x=thetas, label="Q(push left)", title="Q-values vs pole angle θ", xlabel="θ (rad)")
    rlhud.plot(Q[:, 1], x=thetas, label="Q(push right)", title="Q-values vs pole angle θ", xlabel="θ (rad)")

> **🔑 KEY IDEA** · **DQN = Q-learning + a network + stability tricks**
>
> The update is still "move q̂(s, a) toward r + γ max q̂(s′, ·)". Replay decorrelates the data,
> the target network freezes the labels, Huber loss caps the step, and Double DQN reduces
> overestimation. Every trick fights the deadly triad.

## 07 · Debugging deep RL

Deep RL fails silently: no exception, just an agent that doesn't improve. Work through this list.

- **Unit-test the pieces** — gradient check the network; test targets and loss on a 3-row batch you computed by hand (that is what the checks do).
- **Overfit a fixed batch.** A few hundred steps on one minibatch should drive the loss near zero. If not, the problem is the network or the loss, not RL.
- **Log more than the return:** the loss, the mean of max Q, and ε. For CartPole with γ = 0.99, Q should rise toward ~1/(1−γ) = 100. Q values of 10⁶ mean divergence; Q stuck at 0 means no learning.
- **Check `terminated` vs `truncated`** — the most common silent bug.
- **Evaluate greedily**, separately from training. Training returns include exploration noise.
- **Expect instability.** DQN on CartPole often reaches 500 and then collapses for a while ("catastrophic forgetting" when the buffer fills with only good states). Keep the **best checkpoint**, as the provided loop does, and judge algorithms over **several seeds**.
- **Change one thing at a time.** Learning rate and target-sync period are the usual suspects.

In [ ]:
#@title 🧠 **Knowledge check 4** · +25 XP — run, then pick an answer
rlcolab.quiz("09-dqn#3", {"q": "Your DQN's average max-Q climbs past 10⁵ on CartPole (γ = 0.99, reward 1 per step). What is the most likely explanation?", "options": ["The agent has learned a great policy", "The values are diverging — e.g. bootstrapping through terminal states, no target network, or a too-large learning rate", "ε is too small", "The replay buffer is too big"], "answer": 1, "explain": "With reward ≤ 1 per step and γ = 0.99, no true value exceeds 1/(1−γ) = 100. Anything far above that is divergence."})

## 08 · If you have PyTorch

Everything you built by hand maps directly onto PyTorch; autograd replaces `backward()`:

```python
import torch, torch.nn as nn
q = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
q_target = copy.deepcopy(q)
opt = torch.optim.Adam(q.parameters(), lr=1e-3)

S, A, R, S2, D = (torch.as_tensor(x) for x in buffer.sample(64))
with torch.no_grad():                                   # no gradient through the target
    a2 = q(S2).argmax(1, keepdim=True)                  # Double DQN
    y = R + 0.99 * (1 - D) * q_target(S2).gather(1, a2).squeeze(1)
q_sa = q(S).gather(1, A.long().unsqueeze(1)).squeeze(1)
loss = nn.functional.smooth_l1_loss(q_sa, y)            # = Huber with δ = 1
opt.zero_grad(); loss.backward(); opt.step()
```

`torch.no_grad()` is the "semi" in semi-gradient: no gradient flows through the target.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("09-dqn")